In [5]:
import sounddevice as sd
import queue
import json
import sys
import os
import csv
import wave
from datetime import datetime
from vosk import Model, KaldiRecognizer
from IPython.display import clear_output, display

# --- 1. 設定 ---
CSV_FILE = "data/karuta_data.csv"
RESULTS_CSV = "results/recognition_history.csv" 
SAMPLERATE = 16000

# --- 2. 辞書ロジック ---

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    if pos_index == 0 or pos_index >= 2:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    return p

def extract_kanji_parts(text):
    text = text.replace(" ", "")
    p = []
    if len(text) >= 2: p.append(text[:2])
    if len(text) >= 3: p.append(text[:3])
    if len(text) >= 2: p.append(text[-2:])
    if len(text) >= 3: p.append(text[-3:])
    return p

def prepare_karuta_system(file_path):
    if not os.path.exists(file_path):
        print(f"Error: {file_path} が見つかりません。")
        return None, None, None
    raw_cards = []
    with open(file_path, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            words = []; seen = set()
            def add_words(new_words, is_kimari=False):
                for w in new_words:
                    if w in seen: continue
                    if len(w) >= (1 if is_kimari else 2):
                        words.append(w); seen.add(w)
            add_words([kimari], is_kimari=True)
            add_words([gendai.replace(" ", "")])
            k_blocks = kanji.split(); h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): add_words(extract_kanji_parts(k_blocks[i]))
                if i < len(h_blocks): add_words(extract_kana_parts(h_blocks[i], i))
            raw_cards.append({"num": num, "words": words})
    
    if not raw_cards: return None, None, None
    min_count = min(len(c["words"]) for c in raw_cards)
    tag_map = {}; grammar_list = []; card_max_scores = {}
    for c in raw_cards:
        num = c["num"]
        unified_words = c["words"][:min_count]
        total_potential = 0
        for w in unified_words:
            grammar_list.append(w)
            tag_map.setdefault(w, []).append(num)
            total_potential += len(w)
        card_max_scores[num] = total_potential
    return list(set(grammar_list)), tag_map, card_max_scores

# --- 3. 表示と保存の関数 ---

def print_dashboard(scores, max_scores, partial_text, model_name, prev_札, target_札, detail_logs):
    clear_output(wait=True)
    results = []
    for tid, score in scores.items():
        max_s = max_scores.get(tid, 1)
        sim = (score / max_s) * 100
        results.append({"id": tid, "score": score, "max": max_s, "sim": sim})
    top_5 = sorted(results, key=lambda x: x['sim'], reverse=True)[:5]
    output = ["=" * 80, f"設定: [Model: {model_name}] [Prev: {prev_札}] [Target: {target_札}]", 
              "録音中 & リアルタイム類似度判定開始 [停止は中断ボタンをクリック]", "=" * 80]
    clean_partial = partial_text.replace("\n", " ")
    display_partial = clean_partial[-50:] if len(clean_partial) > 50 else clean_partial
    output.append(f" | 認識中の言葉  | {display_partial.ljust(50)}")
    output.extend(["-" * 80, f"{'順位':<4}｜{'推定札':<10} | {'加点/満点':<12}｜{'類似度':<8} ｜棒グラフ", "-" * 80])
    for i in range(5):
        if i < len(top_5):
            res = top_5[i]; bar_count = int(res['sim'] / 5); bar = "■" * bar_count + "□" * (20 - bar_count)
            output.append(f"{i+1}位 ｜ {res['id']:>3}番      | {res['score']:>3} / {res['max']:>3} pts | {res['sim']:>5.1f}%    ｜|{bar}")
        else: output.append(f"{i+1}位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|{'□'*20}")
    
    output.extend(["-" * 80, " 【リアルタイム認識ログ（直近5件）】", "-" * 80])
    if detail_logs:
        recent_logs = detail_logs[-5:][::-1]
        for log in recent_logs:
            output.append(f" [{log[0]}] 単語: {log[1]:<8} ➔ {log[2]:>3}番札に加点 (スコア: {log[3]}/{log[4]}, 類似度: {log[5]})")
    else:
        output.append(" まだキーワードが認識されていません。")
        
    output.extend(["=" * 80])
    print("\n".join(output))

def save_to_result_csv(data):
    os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f) 
        if not file_exists: writer.writerow(["日時", "ファイル名", "前の札", "対象札", "最終推定ID", "最終類似度", "認識単語"])
        writer.writerow(data)

# --- 4. メイン処理 ---

def run_realtime_karuta_with_record():
    print("【使用モデルを選択してください】")
    choice = input("1: Standard (精度重視) / 2: Small (速度重視) -> ")
    model_name = "model_standard" if choice == "1" else "model_small"
    prev_name = input("【前の札名を入力してください】: ")
    target_name = input("【識別したい札名を入力してください】: ")
    
    grammar, tag_map, max_scores = prepare_karuta_system(CSV_FILE)
    if not tag_map: return

    print(f"\n--- モデル読み込み中 ({model_name}) ---")
    model = Model(model_name)
    rec = KaldiRecognizer(model, SAMPLERATE, json.dumps(grammar, ensure_ascii=False))

    dt = datetime.now(); dt_str = dt.strftime("%Y%m%d_%H%M%S")
    base_name = f"{prev_name}_{target_name}_{dt_str}"
    save_filename = f"{base_name}.wav"
    detail_log_filename = f"{base_name}.csv" 
    
    # フォルダの作成を明示的に分ける
    os.makedirs("recordings", exist_ok=True)
    os.makedirs("results", exist_ok=True) # resultsフォルダも作成

    # ★詳細ログCSVの保存先を「results」フォルダに変更
    full_path_wav = os.path.join("recordings", save_filename)
    full_path_csv = os.path.join("results", detail_log_filename) 

    q = queue.Queue(); recorded_frames = [] 
    def callback(indata, frames, time, status):
        data = bytes(indata); q.put(data); recorded_frames.append(data)

    scores = {}; seen_words_per_card = {}; all_detected_words = []
    detail_logs = []

    try:
        with sd.RawInputStream(samplerate=SAMPLERATE, blocksize=4000, 
                               channels=1, dtype='int16', callback=callback):
            while True:
                data = q.get()
                if not rec.AcceptWaveform(data):
                    partial = json.loads(rec.PartialResult()).get('partial', "")
                    if partial:
                        new_words = partial.split()
                        for word in new_words:
                            if word in tag_map:
                                for tid in tag_map[word]:
                                    if tid not in seen_words_per_card: seen_words_per_card[tid] = set()
                                    if word not in seen_words_per_card[tid]:
                                        scores[tid] = scores.get(tid, 0) + len(word)
                                        seen_words_per_card[tid].add(word)
                                        if word not in all_detected_words: all_detected_words.append(word)
                                        
                                        current_sim = (scores[tid] / max_scores.get(tid, 1)) * 100
                                        detail_logs.append([
                                            datetime.now().strftime("%H:%M:%S.%f")[:-3],
                                            word, tid, scores[tid], max_scores.get(tid), f"{current_sim:.1f}%"
                                        ])
                    
                    print_dashboard(scores, max_scores, partial, model_name, prev_name, target_name, detail_logs)

    except KeyboardInterrupt:
        final_id, final_sim = "---", 0.0
        if scores:
            final_id = max(scores, key=lambda k: scores[k]/max_scores.get(k, 1))
            final_sim = (scores[final_id] / max_scores.get(final_id, 1)) * 100

        # WAV保存（recordings/配下）
        with wave.open(full_path_wav, 'wb') as wf:
            wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(SAMPLERATE)
            wf.writeframes(b''.join(recorded_frames))
        
        # 詳細ログCSV保存 (results/配下に保存されるようになりました)
        with open(full_path_csv, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["時刻", "認識単語", "加点札ID", "現在スコア", "満点スコア", "類似度"])
            writer.writerows(detail_logs)

        # 全体履歴CSV保存（results/配下）
        save_to_result_csv([dt.strftime("%Y-%m-%d %H:%M:%S"), save_filename, prev_name, target_name, final_id, f"{final_sim:.1f}%", " ".join(all_detected_words)])
        
        print(f"\n録音保存: {full_path_wav}")
        print(f"詳細ログ保存: {full_path_csv}")
        print(f"最終推定: {final_id} ({final_sim:.1f}%)")

if __name__ == "__main__":
    run_realtime_karuta_with_record()

設定: [Model: model_small] [Prev: nanishi] [Target: harusu]
録音中 & リアルタイム類似度判定開始 [停止は中断ボタンをクリック]
 | 認識中の言葉  | くる ある すぎ 長月 にい けら 白妙                              
--------------------------------------------------------------------------------
順位  ｜推定札        | 加点/満点       ｜類似度      ｜棒グラフ
--------------------------------------------------------------------------------
1位 ｜   2番      |   6 /  46 pts |  13.0%    ｜|■■□□□□□□□□□□□□□□□□□□
2位 ｜  37番      |   2 /  45 pts |   4.4%    ｜|□□□□□□□□□□□□□□□□□□□□
3位 ｜  52番      |   2 /  45 pts |   4.4%    ｜|□□□□□□□□□□□□□□□□□□□□
4位 ｜  40番      |   2 /  46 pts |   4.3%    ｜|□□□□□□□□□□□□□□□□□□□□
5位 ｜  35番      |   2 /  46 pts |   4.3%    ｜|□□□□□□□□□□□□□□□□□□□□
--------------------------------------------------------------------------------
 【リアルタイム認識ログ（直近5件）】
--------------------------------------------------------------------------------
 [15:13:52.955] 単語: 白妙       ➔   4番札に加点 (スコア: 2/46, 類似度: 4.3%)
 [15:13:52.955] 単語: 白妙       ➔   2番札に加点 (スコア: 6/46, 類似度: 13

音声検出＆前の札の下の句の除去

In [10]:
import sounddevice as sd
import queue
import json
import sys
import os
import csv
import wave
import time  # 経過時間測定用に追加
from datetime import datetime
from vosk import Model, KaldiRecognizer
from IPython.display import clear_output, display

# --- 1. 設定 ---
CSV_FILE = "data/karuta_data.csv"
RESULTS_CSV = "results/recognition_history.csv" 
SAMPLERATE = 16000

# ★【新規追加】前の札の下の句を無視する時間（秒）
# 小数点第一位までで自由に変更可能です
SKIP_SECONDS = 6.0

# --- 2. 辞書ロジック ---

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    if pos_index == 0 or pos_index >= 2:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    return p

def extract_kanji_parts(text):
    text = text.replace(" ", "")
    p = []
    if len(text) >= 2: p.append(text[:2])
    if len(text) >= 3: p.append(text[:3])
    if len(text) >= 2: p.append(text[-2:])
    if len(text) >= 3: p.append(text[-3:])
    return p

def prepare_karuta_system(file_path):
    if not os.path.exists(file_path):
        print(f"Error: {file_path} が見つかりません。")
        return None, None, None
    raw_cards = []
    with open(file_path, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            words = []; seen = set()
            def add_words(new_words, is_kimari=False):
                for w in new_words:
                    if w in seen: continue
                    if len(w) >= (1 if is_kimari else 2):
                        words.append(w); seen.add(w)
            add_words([kimari], is_kimari=True)
            add_words([gendai.replace(" ", "")])
            k_blocks = kanji.split(); h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): add_words(extract_kanji_parts(k_blocks[i]))
                if i < len(h_blocks): add_words(extract_kana_parts(h_blocks[i], i))
            raw_cards.append({"num": num, "words": words})
    
    if not raw_cards: return None, None, None
    min_count = min(len(c["words"]) for c in raw_cards)
    tag_map = {}; grammar_list = []; card_max_scores = {}
    for c in raw_cards:
        num = c["num"]
        unified_words = c["words"][:min_count]
        total_potential = 0
        for w in unified_words:
            grammar_list.append(w)
            tag_map.setdefault(w, []).append(num)
            total_potential += len(w)
        card_max_scores[num] = total_potential
    return list(set(grammar_list)), tag_map, card_max_scores

# --- 3. 表示と保存の関数 ---

# 引数に「is_skipping」と「elapsed」を追加
def print_dashboard(scores, max_scores, partial_text, model_name, prev_札, target_札, detail_logs, is_skipping, elapsed):
    clear_output(wait=True)
    results = []
    for tid, score in scores.items():
        max_s = max_scores.get(tid, 1)
        sim = (score / max_s) * 100
        results.append({"id": tid, "score": score, "max": max_s, "sim": sim})
    top_5 = sorted(results, key=lambda x: x['sim'], reverse=True)[:5]
    
    # 状態表示のラベルを作成
    if is_skipping:
        status_label = f"【下の句スキップ中: {elapsed:.1f}s / {SKIP_SECONDS:.1f}s】(加点無効)"
    else:
        status_label = f"【類似度判定中: {elapsed:.1f}s経過】(加点有効)"

    output = ["=" * 80, 
              f"設定: [Model: {model_name}] [Prev: {prev_札}] [Target: {target_札}]", 
              f"状況: {status_label}", "=" * 80]
              
    clean_partial = partial_text.replace("\n", " ")
    display_partial = clean_partial[-50:] if len(clean_partial) > 50 else clean_partial
    output.append(f" | 認識中の言葉  | {display_partial.ljust(50)}")
    output.extend(["-" * 80, f"{'順位':<4}｜{'推定札':<10} | {'加点/満点':<12}｜{'類似度':<8} ｜棒グラフ", "-" * 80])
    for i in range(5):
        if i < len(top_5):
            res = top_5[i]; bar_count = int(res['sim'] / 5); bar = "■" * bar_count + "□" * (20 - bar_count)
            output.append(f"{i+1}位 ｜ {res['id']:>3}番      | {res['score']:>3} / {res['max']:>3} pts | {res['sim']:>5.1f}%    ｜|{bar}")
        else: output.append(f"{i+1}位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|{'□'*20}")
    
    output.extend(["-" * 80, " 【リアルタイム認識ログ（直近5件）】", "-" * 80])
    if detail_logs:
        recent_logs = detail_logs[-5:][::-1]
        for log in recent_logs:
            output.append(f" [{log[0]}] 単語: {log[1]:<8} ➔ {log[2]:>3}番札に加点 (スコア: {log[3]}/{log[4]}, 類似度: {log[5]})")
    else:
        output.append(" まだキーワードが認識されていません。")
        
    output.extend(["=" * 80])
    print("\n".join(output))

def save_to_result_csv(data):
    os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f) 
        if not file_exists: writer.writerow(["日時", "ファイル名", "前の札", "対象札", "最終推定ID", "最終類似度", "認識単語"])
        writer.writerow(data)

# --- 4. メイン処理 ---

def run_realtime_karuta_with_record():
    print("【使用モデルを選択してください】")
    choice = input("1: Standard (精度重視) / 2: Small (速度重視) -> ")
    model_name = "model_standard" if choice == "1" else "model_small"
    prev_name = input("【前の札名を入力してください】: ")
    target_name = input("【識別したい札名を入力してください】: ")
    
    grammar, tag_map, max_scores = prepare_karuta_system(CSV_FILE)
    if not tag_map: return

    print(f"\n--- モデル読み込み中 ({model_name}) ---")
    model = Model(model_name)
    rec = KaldiRecognizer(model, SAMPLERATE, json.dumps(grammar, ensure_ascii=False))

    dt = datetime.now(); dt_str = dt.strftime("%Y%m%d_%H%M%S")
    base_name = f"{prev_name}_{target_name}_{dt_str}"
    save_filename = f"{base_name}.wav"
    detail_log_filename = f"{base_name}.csv" 
    
    os.makedirs("recordings", exist_ok=True)
    os.makedirs("results", exist_ok=True)

    full_path_wav = os.path.join("recordings", save_filename)
    full_path_csv = os.path.join("results", detail_log_filename) 

    q = queue.Queue(); recorded_frames = [] 
    def callback(indata, frames, time, status):
        data = bytes(indata); q.put(data); recorded_frames.append(data)

    scores = {}; seen_words_per_card = {}; all_detected_words = []
    detail_logs = []

    # ★音声ストリームに入る直前に、開始時刻を記録
    start_time = time.time()

    try:
        with sd.RawInputStream(samplerate=SAMPLERATE, blocksize=4000, 
                               channels=1, dtype='int16', callback=callback):
            while True:
                data = q.get()
                
                # ★現在の経過時間を計算
                elapsed = time.time() - start_time
                is_skipping = (elapsed < SKIP_SECONDS)

                if not rec.AcceptWaveform(data):
                    partial = json.loads(rec.PartialResult()).get('partial', "")
                    if partial:
                        new_words = partial.split()
                        for word in new_words:
                            # ★スキップ時間（6.0秒）を過ぎている場合のみ、加点処理を行う
                            if not is_skipping:
                                if word in tag_map:
                                    for tid in tag_map[word]:
                                        if tid not in seen_words_per_card: seen_words_per_card[tid] = set()
                                        if word not in seen_words_per_card[tid]:
                                            scores[tid] = scores.get(tid, 0) + len(word)
                                            seen_words_per_card[tid].add(word)
                                            if word not in all_detected_words: all_detected_words.append(word)
                                            
                                            current_sim = (scores[tid] / max_scores.get(tid, 1)) * 100
                                            detail_logs.append([
                                                datetime.now().strftime("%H:%M:%S.%f")[:-3],
                                                word, tid, scores[tid], max_scores.get(tid), f"{current_sim:.1f}%"
                                            ])
                    
                    # ダッシュボード表示に「is_skipping」と「elapsed」の状態を渡す
                    print_dashboard(scores, max_scores, partial, model_name, prev_name, target_name, detail_logs, is_skipping, elapsed)

    except KeyboardInterrupt:
        final_id, final_sim = "---", 0.0
        if scores:
            final_id = max(scores, key=lambda k: scores[k]/max_scores.get(k, 1))
            final_sim = (scores[final_id] / max_scores.get(final_id, 1)) * 100

        # WAV保存（recordings/配下）
        with wave.open(full_path_wav, 'wb') as wf:
            wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(SAMPLERATE)
            wf.writeframes(b''.join(recorded_frames))
        
        # 詳細ログCSV保存 (results/配下)
        with open(full_path_csv, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["時刻", "認識単語", "加点札ID", "現在スコア", "満点スコア", "類似度"])
            writer.writerows(detail_logs)

        # 全体履歴CSV保存（results/配下）
        save_to_result_csv([dt.strftime("%Y-%m-%d %H:%M:%S"), save_filename, prev_name, target_name, final_id, f"{final_sim:.1f}%", " ".join(all_detected_words)])
        
        print(f"\n録音保存: {full_path_wav}")
        print(f"詳細ログ保存: {full_path_csv}")
        print(f"最終推定: {final_id} ({final_sim:.1f}%)")

if __name__ == "__main__":
    run_realtime_karuta_with_record()

設定: [Model: model_small] [Prev: nanishi] [Target: haruno]
状況: 【類似度判定中: 17.6s経過】(加点有効)
 | 認識中の言葉  | 春の ゆめ ばか 山桜                                       
--------------------------------------------------------------------------------
順位  ｜推定札        | 加点/満点       ｜類似度      ｜棒グラフ
--------------------------------------------------------------------------------
1位 ｜  67番      |   8 /  46 pts |  17.4%    ｜|■■■□□□□□□□□□□□□□□□□□
2位 ｜  33番      |   4 /  45 pts |   8.9%    ｜|■□□□□□□□□□□□□□□□□□□□
3位 ｜  15番      |   4 /  49 pts |   8.2%    ｜|■□□□□□□□□□□□□□□□□□□□
4位 ｜  21番      |   3 /  46 pts |   6.5%    ｜|■□□□□□□□□□□□□□□□□□□□
5位 ｜  12番      |   2 /  45 pts |   4.4%    ｜|□□□□□□□□□□□□□□□□□□□□
--------------------------------------------------------------------------------
 【リアルタイム認識ログ（直近5件）】
--------------------------------------------------------------------------------
 [15:53:42.595] 単語: 山桜       ➔  66番札に加点 (スコア: 2/45, 類似度: 4.4%)
 [15:53:42.075] 単語: あま       ➔  90番札に加点 (スコア: 2/46, 類似度: 4.3%)
 [15

辞書内の単語が使われたら音声検出ってしてる。ただこれは、下の句がCSVファイルに含まれていないので、うまくいかないときもある

In [12]:
import sounddevice as sd
import queue
import json
import sys
import os
import csv
import wave
import time  # 経過時間測定用
from datetime import datetime
from vosk import Model, KaldiRecognizer
from IPython.display import clear_output, display

# --- 1. 設定 ---
CSV_FILE = "data/karuta_data.csv"
RESULTS_CSV = "results/recognition_history.csv" 
SAMPLERATE = 16000

# 前の札の下の句を無視する時間（秒）
SKIP_SECONDS = 6.0

# --- 2. 辞書ロジック ---

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    if pos_index == 0 or pos_index >= 2:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    return p

def extract_kanji_parts(text):
    text = text.replace(" ", "")
    p = []
    if len(text) >= 2: p.append(text[:2])
    if len(text) >= 3: p.append(text[:3])
    if len(text) >= 2: p.append(text[-2:])
    if len(text) >= 3: p.append(text[-3:])
    return p

def prepare_karuta_system(file_path):
    if not os.path.exists(file_path):
        print(f"Error: {file_path} が見つかりません。")
        return None, None, None
    raw_cards = []
    with open(file_path, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            words = []; seen = set()
            def add_words(new_words, is_kimari=False):
                for w in new_words:
                    if w in seen: continue
                    if len(w) >= (1 if is_kimari else 2):
                        words.append(w); seen.add(w)
            add_words([kimari], is_kimari=True)
            add_words([gendai.replace(" ", "")])
            k_blocks = kanji.split(); h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): add_words(extract_kanji_parts(k_blocks[i]))
                if i < len(h_blocks): add_words(extract_kana_parts(h_blocks[i], i))
            raw_cards.append({"num": num, "words": words})
    
    if not raw_cards: return None, None, None
    min_count = min(len(c["words"]) for c in raw_cards)
    tag_map = {}; grammar_list = []; card_max_scores = {}
    for c in raw_cards:
        num = c["num"]
        unified_words = c["words"][:min_count]
        total_potential = 0
        for w in unified_words:
            grammar_list.append(w)
            tag_map.setdefault(w, []).append(num)
            total_potential += len(w)
        card_max_scores[num] = total_potential
    return list(set(grammar_list)), tag_map, card_max_scores

# --- 3. 表示と保存の関数 ---

# 引数「is_skipping」を状態を表す文字列「status_mode」に変更
def print_dashboard(scores, max_scores, partial_text, model_name, prev_札, target_札, detail_logs, status_mode, elapsed):
    clear_output(wait=True)
    results = []
    for tid, score in scores.items():
        max_s = max_scores.get(tid, 1)
        sim = (score / max_s) * 100
        results.append({"id": tid, "score": score, "max": max_s, "sim": sim})
    top_5 = sorted(results, key=lambda x: x['sim'], reverse=True)[:5]
    
    # 状況表示ラベルの作成
    if status_mode == "wait":
        status_label = "【音声検出待ち...】(声が入るとカウントを開始します)"
    elif status_mode == "skip":
        status_label = f"【下の句スキップ中: {elapsed:.1f}s / {SKIP_SECONDS:.1f}s】(加点無効)"
    else:
        status_label = f"【類似度判定中: {elapsed:.1f}s経過】(加点有効)"

    output = ["=" * 80, 
              f"設定: [Model: {model_name}] [Prev: {prev_札}] [Target: {target_札}]", 
              f"状況: {status_label}", "=" * 80]
              
    clean_partial = partial_text.replace("\n", " ")
    display_partial = clean_partial[-50:] if len(clean_partial) > 50 else clean_partial
    output.append(f" | 認識中の言葉  | {display_partial.ljust(50)}")
    output.extend(["-" * 80, f"{'順位':<4}｜{'推定札':<10} | {'加点/満点':<12}｜{'類似度':<8} ｜棒グラフ", "-" * 80])
    for i in range(5):
        if i < len(top_5):
            res = top_5[i]; bar_count = int(res['sim'] / 5); bar = "■" * bar_count + "□" * (20 - bar_count)
            output.append(f"{i+1}位 ｜ {res['id']:>3}番      | {res['score']:>3} / {res['max']:>3} pts | {res['sim']:>5.1f}%    ｜|{bar}")
        else: output.append(f"{i+1}位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|{'□'*20}")
    
    output.extend(["-" * 80, " 【リアルタイム認識ログ（直近5件）】", "-" * 80])
    if detail_logs:
        recent_logs = detail_logs[-5:][::-1]
        for log in recent_logs:
            output.append(f" [{log[0]}] 単語: {log[1]:<8} ➔ {log[2]:>3}番札に加点 (スコア: {log[3]}/{log[4]}, 類似度: {log[5]})")
    else:
        output.append(" まだキーワードが認識されていません。")
        
    output.extend(["=" * 80])
    print("\n".join(output))

def save_to_result_csv(data):
    os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f) 
        if not file_exists: writer.writerow(["日時", "ファイル名", "前の札", "対象札", "最終推定ID", "最終類似度", "認識単語"])
        writer.writerow(data)

# --- 4. メイン処理 ---

def run_realtime_karuta_with_record():
    print("【使用モデルを選択してください】")
    choice = input("1: Standard (精度重視) / 2: Small (速度重視) -> ")
    model_name = "model_standard" if choice == "1" else "model_small"
    prev_name = input("【前の札名を入力してください】: ")
    target_name = input("【識別したい札名を入力してください】: ")
    
    grammar, tag_map, max_scores = prepare_karuta_system(CSV_FILE)
    if not tag_map: return

    print(f"\n--- モデル読み込み中 ({model_name}) ---")
    model = Model(model_name)
    rec = KaldiRecognizer(model, SAMPLERATE, json.dumps(grammar, ensure_ascii=False))

    dt = datetime.now(); dt_str = dt.strftime("%Y%m%d_%H%M%S")
    base_name = f"{prev_name}_{target_name}_{dt_str}"
    save_filename = f"{base_name}.wav"
    detail_log_filename = f"{base_name}.csv" 
    
    os.makedirs("recordings", exist_ok=True)
    os.makedirs("results", exist_ok=True)

    full_path_wav = os.path.join("recordings", save_filename)
    full_path_csv = os.path.join("results", detail_log_filename) 

    q = queue.Queue(); recorded_frames = [] 
    def callback(indata, frames, time, status):
        data = bytes(indata); q.put(data); recorded_frames.append(data)

    scores = {}; seen_words_per_card = {}; all_detected_words = []
    detail_logs = []

    # 音声検出のトリガー用変数
    audio_detected_time = None  # 音声を最初に検知した時刻

    try:
        with sd.RawInputStream(samplerate=SAMPLERATE, blocksize=4000, 
                               channels=1, dtype='int16', callback=callback):
            while True:
                data = q.get()
                
                # Voskにデータを送りつつ、現在の状態を取得
                is_recognized = rec.AcceptWaveform(data)
                partial = json.loads(rec.PartialResult()).get('partial', "")

                # ★【音声検出ロジック】部分認識（partial）に何か文字が入った瞬間をスタートとする
                if audio_detected_time is None and partial.strip() != "":
                    audio_detected_time = time.time()

                # 時間とモードの判定
                if audio_detected_time is None:
                    # まだ声が検出されていない状態
                    elapsed = 0.0
                    status_mode = "wait"
                    is_skipping = True
                else:
                    # 声を検出した後の経過時間
                    elapsed = time.time() - audio_detected_time
                    if elapsed < SKIP_SECONDS:
                        status_mode = "skip"
                        is_skipping = True
                    else:
                        status_mode = "run"
                        is_skipping = False

                if not is_recognized:
                    if partial:
                        new_words = partial.split()
                        for word in new_words:
                            # スキップ時間（6秒間）が終わるまでは加点しない
                            if not is_skipping:
                                if word in tag_map:
                                    for tid in tag_map[word]:
                                        if tid not in seen_words_per_card: seen_words_per_card[tid] = set()
                                        if word not in seen_words_per_card[tid]:
                                            scores[tid] = scores.get(tid, 0) + len(word)
                                            seen_words_per_card[tid].add(word)
                                            if word not in all_detected_words: all_detected_words.append(word)
                                            
                                            current_sim = (scores[tid] / max_scores.get(tid, 1)) * 100
                                            detail_logs.append([
                                                datetime.now().strftime("%H:%M:%S.%f")[:-3],
                                                word, tid, scores[tid], max_scores.get(tid), f"{current_sim:.1f}%"
                                            ])
                    
                    # ダッシュボード表示を更新
                    print_dashboard(scores, max_scores, partial, model_name, prev_name, target_name, detail_logs, status_mode, elapsed)

    except KeyboardInterrupt:
        final_id, final_sim = "---", 0.0
        if scores:
            final_id = max(scores, key=lambda k: scores[k]/max_scores.get(k, 1))
            final_sim = (scores[final_id] / max_scores.get(final_id, 1)) * 100

        # WAV保存（recordings/配下）
        with wave.open(full_path_wav, 'wb') as wf:
            wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(SAMPLERATE)
            wf.writeframes(b''.join(recorded_frames))
        
        # 詳細ログCSV保存 (results/配下)
        with open(full_path_csv, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["時刻", "認識単語", "加点札ID", "現在スコア", "満点スコア", "類似度"])
            writer.writerows(detail_logs)

        # 全体履歴CSV保存（results/配下）
        save_to_result_csv([dt.strftime("%Y-%m-%d %H:%M:%S"), save_filename, prev_name, target_name, final_id, f"{final_sim:.1f}%", " ".join(all_detected_words)])
        
        print(f"\n録音保存: {full_path_wav}")
        print(f"詳細ログ保存: {full_path_csv}")
        print(f"最終推定: {final_id} ({final_sim:.1f}%)")

if __name__ == "__main__":
    run_realtime_karuta_with_record()

設定: [Model: model_small] [Prev: haruno] [Target: harusu]
状況: 【下の句スキップ中: 4.3s / 6.0s】(加点無効)
 | 認識中の言葉  | すぎ 月見 白妙                                          
--------------------------------------------------------------------------------
順位  ｜推定札        | 加点/満点       ｜類似度      ｜棒グラフ
--------------------------------------------------------------------------------
1位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|□□□□□□□□□□□□□□□□□□□□
2位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|□□□□□□□□□□□□□□□□□□□□
3位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|□□□□□□□□□□□□□□□□□□□□
4位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|□□□□□□□□□□□□□□□□□□□□
5位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|□□□□□□□□□□□□□□□□□□□□
--------------------------------------------------------------------------------
 【リアルタイム認識ログ（直近5件）】
--------------------------------------------------------------------------------
 まだキーワードが認識されていません。

録音保存: recordings\haruno_harusu_20260517_155607.wav
詳細ログ保存: results\haruno_harusu_20260517_155607.csv
最終推

こっちはSileroVAD＋時間除去を用いた音声検出　これを使う

In [8]:
import sounddevice as sd
import queue
import json
import sys
import os
import csv
import wave
import time
import numpy as np
import torch
import threading  # ★リアルタイムタイマー用に追記
from datetime import datetime
from vosk import Model, KaldiRecognizer
from IPython.display import clear_output, display

# --- 1. 設定 ---
CSV_FILE = "data/karuta_data.csv"
RESULTS_CSV = "results/recognition_history.csv" 
SAMPLERATE = 16000

# 前の札の下の句を無視する時間（秒）
SKIP_SECONDS = 8.0

# VADの判定しきい値（0.0 〜 1.0）
VAD_THRESHOLD = 0.5

# --- 2. 辞書ロジック ---

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    if pos_index == 0 or pos_index >= 2:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    return p

def extract_kanji_parts(text):
    text = text.replace(" ", "")
    p = []
    if len(text) >= 2: p.append(text[:2])
    if len(text) >= 3: p.append(text[:3])
    if len(text) >= 2: p.append(text[-2:])
    if len(text) >= 3: p.append(text[-3:])
    return p

def prepare_karuta_system(file_path):
    if not os.path.exists(file_path):
        print(f"Error: {file_path} が見つかりません。")
        return None, None, None
    raw_cards = []
    with open(file_path, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            words = []; seen = set()
            def add_words(new_words, is_kimari=False):
                for w in new_words:
                    if w in seen: continue
                    if len(w) >= (1 if is_kimari else 2):
                        words.append(w); seen.add(w)
            add_words([kimari], is_kimari=True)
            add_words([gendai.replace(" ", "")])
            k_blocks = kanji.split(); h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): add_words(extract_kanji_parts(k_blocks[i]))
                if i < len(h_blocks): add_words(extract_kana_parts(h_blocks[i], i))
            raw_cards.append({"num": num, "words": words})
    
    if not raw_cards: return None, None, None
    
    tag_map = {}; grammar_list = []; card_max_scores = {}
    for c in raw_cards:
        num = c["num"]
        unified_words = c["words"]
        total_potential = 0
        for w in unified_words:
            grammar_list.append(w)
            tag_map.setdefault(w, []).append(num)
            total_potential += len(w)
        card_max_scores[num] = total_potential
        
    return list(set(grammar_list)), tag_map, card_max_scores

# --- 3. 表示と保存の関数 ---

def print_dashboard(scores, max_scores, partial_text, model_name, prev_札, target_札, detail_logs, status_mode, elapsed, speech_prob):
    clear_output(wait=True)
    results = []
    for tid, score in scores.items():
        max_s = max_scores.get(tid, 1)
        sim = (score / max_s) * 100
        results.append({"id": tid, "score": score, "max": max_s, "sim": sim})
    top_5 = sorted(results, key=lambda x: x['sim'], reverse=True)[:5]
    
    # 状態表示
    if status_mode == "wait":
        status_label = f"【音声検出待ち...】(VAD音声確率: {speech_prob*100:.1f}% / しきい値: {VAD_THRESHOLD*100:.1f}%)"
    elif status_mode == "skip":
        status_label = f"【下の句スキップ中: {elapsed:.1f}s / {SKIP_SECONDS:.1f}s】(加点無効)"
    else:
        status_label = f"【類似度判定中: {elapsed:.1f}s経過】(加点有効)"

    output = ["=" * 80, 
              f"設定: [Model: {model_name}] [Prev: {prev_札}] [Target: {target_札}]", 
              f"状況: {status_label}", "=" * 80]
              
    clean_partial = partial_text.replace("\n", " ")
    display_partial = clean_partial[-50:] if len(clean_partial) > 50 else clean_partial
    output.append(f" | 認識中の言葉  | {display_partial.ljust(50)}")
    output.extend(["-" * 80, f"{'順位':<4}｜{'推定札':<10} | {'加点/満点':<12}｜{'類似度':<8} ｜棒グラフ", "-" * 80])
    for i in range(5):
        if i < len(top_5):
            res = top_5[i]; bar_count = int(res['sim'] / 5); bar = "■" * bar_count + "□" * (20 - bar_count)
            output.append(f"{i+1}位 ｜ {res['id']:>3}番      | {res['score']:>3} / {res['max']:>3} pts | {res['sim']:>5.1f}%    ｜|{bar}")
        else: output.append(f"{i+1}位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|{'□'*20}")
    
    output.extend(["-" * 80, " 【リアルタイム認識ログ（直近5件）】", "-" * 80])
    if detail_logs:
        recent_logs = detail_logs[-5:][::-1]
        for log in recent_logs:
            output.append(f" [{log[0]}] 単語: {log[1]:<8} ➔ {log[2]:>3}番札に加点 (スコア: {log[3]}/{log[4]}, 類似度: {log[5]})")
    else:
        output.append(" まだキーワードが認識されていません。")
        
    output.extend(["=" * 80])
    print("\n".join(output))

def save_to_result_csv(data):
    os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f) 
        if not file_exists: writer.writerow(["日時", "ファイル名", "前の札", "対象札", "最終推定ID", "最終類似度", "認識単語"])
        writer.writerow(data)

# --- 4. メイン処理 ---

def run_realtime_karuta_with_record():
    print("【使用モデルを選択してください】")
    choice = input("1: Standard (精度重視) / 2: Small (速度重視) -> ")
    model_name = "model_standard" if choice == "1" else "model_small"
    prev_name = input("【前の札名を入力してください】: ")
    target_name = input("【識別したい札名を入力してください】: ")
    
    grammar, tag_map, max_scores = prepare_karuta_system(CSV_FILE)
    if not tag_map: return

    # Silero VAD モデルの読み込み
    print("\n--- Silero VAD 読み込み中 ---")
    vad_model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                                      model='silero_vad',
                                      force_reload=False,
                                      trust_repo=True)
    (get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

    print(f"\n--- Vosk モデル読み込み中 ({model_name}) ---")
    model = Model(model_name)
    rec = KaldiRecognizer(model, SAMPLERATE, json.dumps(grammar, ensure_ascii=False))

    dt = datetime.now(); dt_str = dt.strftime("%Y%m%d_%H%M%S")
    base_name = f"{prev_name}_{target_name}_{dt_str}"
    save_filename = f"{base_name}.wav"
    detail_log_filename = f"{base_name}.csv" 
    
    os.makedirs("recordings", exist_ok=True)
    os.makedirs("results", exist_ok=True)

    full_path_wav = os.path.join("recordings", save_filename)
    full_path_csv = os.path.join("results", detail_log_filename) 

    q = queue.Queue(); recorded_frames = [] 
    def callback(indata, frames, time, status):
        data = bytes(indata); q.put(data); recorded_frames.append(data)

    # スレッド間で共有する状態オブジェクト
    state = {
        "scores": {},
        "seen_words": {},
        "all_words": [],
        "detail_logs": [],
        "result_text": "",
        "audio_detected_time": None,
        "speech_prob": 0.0,
        "status_mode": "wait",
        "elapsed": 0.0,
        "is_running": True
    }

    # ★ 0.1秒ごとに無音状態に関係なく画面を強制リフレッシュするスレッド用の関数
    def dashboard_refresh_worker():
        while state["is_running"]:
            if state["audio_detected_time"] is not None:
                # 実際の時間軸を基準に経過時間をミリ秒単位でリアルタイム更新
                state["elapsed"] = time.time() - state["audio_detected_time"]
                
                if state["elapsed"] < SKIP_SECONDS:
                    state["status_mode"] = "skip"
                else:
                    state["status_mode"] = "run"
            else:
                state["elapsed"] = 0.0
                state["status_mode"] = "wait"
                
            # ダッシュボード表示を更新
            print_dashboard(
                state["scores"], max_scores, state["result_text"], 
                model_name, prev_name, target_name, state["detail_logs"], 
                state["status_mode"], state["elapsed"], state["speech_prob"]
            )
            time.sleep(0.1)  # 10Hzの周期でリフレッシュ

    # 画面更新スレッドをバックグラウンドで開始
    refresh_thread = threading.Thread(target=dashboard_refresh_worker, daemon=True)
    refresh_thread.start()

    try:
        # blocksize=4000の安定したバッファサイズでマイク入力を開始
        with sd.RawInputStream(samplerate=SAMPLERATE, blocksize=4000, 
                               channels=1, dtype='int16', callback=callback):
            while True:
                data = q.get()
                
                # --- 1. Voskによる音声認識 ---
                if not rec.AcceptWaveform(data):
                    state["result_text"] = json.loads(rec.PartialResult()).get('partial', "")
                else:
                    state["result_text"] = json.loads(rec.Result()).get('text', "")

                # --- 2. Silero VAD による音声解析（4000を512ずつ小分け） ---
                audio_data = np.frombuffer(data, dtype=np.int16).astype(np.float32) / 32768.0
                chunk_size = 512
                current_max_prob = 0.0
                
                for i in range(0, len(audio_data), chunk_size):
                    chunk = audio_data[i:i + chunk_size]
                    if len(chunk) < chunk_size:
                        continue
                    audio_tensor = torch.from_numpy(chunk)
                    prob = vad_model(audio_tensor, SAMPLERATE).item()
                    if prob > current_max_prob:
                        current_max_prob = prob
                
                state["speech_prob"] = current_max_prob

                # --- 3. タイマーのトリガー判定 ---
                if state["audio_detected_time"] is None and state["speech_prob"] > VAD_THRESHOLD:
                    state["audio_detected_time"] = time.time()

                # --- 4. 加点処理 ---
                if state["result_text"]:
                    new_words = state["result_text"].split()
                    for word in new_words:
                        # スレッド側が判定した最新のstatus_modeを参照
                        if state["status_mode"] == "run":
                            if word in tag_map:
                                for tid in tag_map[word]:
                                    if tid not in state["seen_words"]: 
                                        state["seen_words"][tid] = set()
                                    if word not in state["seen_words"][tid]:
                                        state["scores"][tid] = state["scores"].get(tid, 0) + len(word)
                                        state["seen_words"][tid].add(word)
                                        if word not in state["all_words"]: 
                                            state["all_words"].append(word)
                                        
                                        current_sim = (state["scores"][tid] / max_scores.get(tid, 1)) * 100
                                        state["detail_logs"].append([
                                            datetime.now().strftime("%H:%M:%S.%f")[:-3],
                                            word, tid, state["scores"][tid], max_scores.get(tid), f"{current_sim:.1f}%"
                                        ])

    except KeyboardInterrupt:
        state["is_running"] = False  # リフレッシュスレッドを停止
        
        final_id, final_sim = "---", 0.0
        if state["scores"]:
            final_id = max(state["scores"], key=lambda k: state["scores"][k]/max_scores.get(k, 1))
            final_sim = (state["scores"][final_id] / max_scores.get(final_id, 1)) * 100

        # WAV保存
        with wave.open(full_path_wav, 'wb') as wf:
            wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(SAMPLERATE)
            wf.writeframes(b''.join(recorded_frames))
        
        # 詳細ログCSV保存
        with open(full_path_csv, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["時刻", "認識単語", "加点札ID", "現在スコア", "満点スコア", "類似度"])
            writer.writerows(state["detail_logs"])

        # 全体履歴CSV保存
        save_to_result_csv([dt.strftime("%Y-%m-%d %H:%M:%S"), save_filename, prev_name, target_name, final_id, f"{final_sim:.1f}%", " ".join(state["all_words"])])
        
        print(f"\n録音保存: {full_path_wav}")
        print(f"詳細ログ保存: {full_path_csv}")
        print(f"最終推定: {final_id} ({final_sim:.1f}%)")

if __name__ == "__main__":
    run_realtime_karuta_with_record()

設定: [Model: model_small] [Prev: hu] [Target: se]
状況: 【類似度判定中: 27.5s経過】(加点有効)
 | 認識中の言葉  |                                                   
--------------------------------------------------------------------------------
順位  ｜推定札        | 加点/満点       ｜類似度      ｜棒グラフ
--------------------------------------------------------------------------------
1位 ｜  40番      |   2 /  48 pts |   4.2%    ｜|□□□□□□□□□□□□□□□□□□□□
2位 ｜  77番      |   2 /  50 pts |   4.0%    ｜|□□□□□□□□□□□□□□□□□□□□
3位 ｜  15番      |   2 /  61 pts |   3.3%    ｜|□□□□□□□□□□□□□□□□□□□□
4位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|□□□□□□□□□□□□□□□□□□□□
5位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|□□□□□□□□□□□□□□□□□□□□
--------------------------------------------------------------------------------
 【リアルタイム認識ログ（直近5件）】
--------------------------------------------------------------------------------
 [17:14:39.236] 単語: にい       ➔  40番札に加点 (スコア: 2/48, 類似度: 4.2%)
 [17:14:39.236] 単語: にい       ➔  15番札に加点 (スコア: 2/61, 類似度: 3.3%)
 [17:14:37.24

辞書内の共通語を単語、札番号ごとに調べる

In [3]:
import csv
import os

# --- 設定 ---
INPUT_CSV = "data/karuta_data.csv"
OUTPUT_DICTIONARY_CSV = "results/karuta_dictionary_map.csv"

# --- 1. 元のコードから移植した切り出しロジック ---

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    if pos_index == 0 or pos_index >= 2:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    return p

def extract_kanji_parts(text):
    text = text.replace(" ", "")
    p = []
    if len(text) >= 2: p.append(text[:2])
    if len(text) >= 3: p.append(text[:3])
    if len(text) >= 2: p.append(text[-2:])
    if len(text) >= 3: p.append(text[-3:])
    return p

# --- 2. 辞書抽出とCSV出力メイン処理 ---

def export_karuta_dictionary_sorted_by_count():
    if not os.path.exists(INPUT_CSV):
        print(f"Error: 入力ファイル {INPUT_CSV} が見つかりません。")
        return

    raw_cards = []
    
    # 1. 元ファイルからデータを読み込み、パーツ化
    with open(INPUT_CSV, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            
            words = []
            seen = set()
            
            def add_words(new_words, is_kimari=False):
                for w in new_words:
                    if w in seen: continue
                    if len(w) >= (1 if is_kimari else 2):
                        words.append(w)
                        seen.add(w)
                        
            add_words([kimari], is_kimari=True)
            add_words([gendai.replace(" ", "")])
            
            k_blocks = kanji.split()
            h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): add_words(extract_kanji_parts(k_blocks[i]))
                if i < len(h_blocks): add_words(extract_kana_parts(h_blocks[i], i))
                
            raw_cards.append({"num": num, "words": words})

    # 2. 抽出された単語ごとに、対応する札番号を集計 (逆引きマップの作成)
    tag_map = {}
    for c in raw_cards:
        num = c["num"]
        for w in c["words"]:
            tag_map.setdefault(w, []).append(num)

    # 3. 集計結果をCSVファイルに出力
    os.makedirs(os.path.dirname(OUTPUT_DICTIONARY_CSV), exist_ok=True)
    
    with open(OUTPUT_DICTIONARY_CSV, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        # ヘッダーの書き込み
        writer.writerow(["抽出単語（キーワード）", "文字数", "対応する札番号リスト", "該当する札の総数"])
        
        # ★【変更箇所】該当する札の総数（リストの長さ）が多い順にソート。
        # 同数の場合は、文字数が長いものを優先。
        sorted_words = sorted(
            tag_map.keys(), 
            key=lambda x: (len(tag_map[x]), len(x)), 
            reverse=True
        )
        
        for word in sorted_words:
            card_numbers = tag_map[word]
            card_numbers_str = " ".join(card_numbers) 
            
            writer.writerow([
                word, 
                len(word), 
                card_numbers_str, 
                len(card_numbers)
            ])

    print(f"成功: 総数の多い順に並び替えて {OUTPUT_DICTIONARY_CSV} に出力しました。")
    print(f"総登録単語数: {len(tag_map)} 語")

if __name__ == "__main__":
    export_karuta_dictionary_sorted_by_count()

成功: 総数の多い順に並び替えて results/karuta_dictionary_map.csv に出力しました。
総登録単語数: 1695 語


これは札番号ごとの共通語の可視化

In [8]:
import csv
import os

# --- 設定 ---
INPUT_CSV = "data/karuta_data.csv"
OUTPUT_CARD_SUMMARY_CSV = "results/karuta_card_overlap_summary.csv"

# --- 1. 元のコードから移植した切り出しロジック ---

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    if pos_index == 0 or pos_index >= 2:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    return p

def extract_kanji_parts(text):
    text = text.replace(" ", "")
    p = []
    if len(text) >= 2: p.append(text[:2])
    if len(text) >= 3: p.append(text[:3])
    if len(text) >= 2: p.append(text[-2:])
    if len(text) >= 3: p.append(text[-3:])
    return p

# --- 2. 集計とCSV出力メイン処理 ---

def export_card_overlap_summary():
    if not os.path.exists(INPUT_CSV):
        print(f"Error: 入力ファイル {INPUT_CSV} が見つかりません。")
        return

    raw_cards = []
    
    # 1. 元ファイルからデータを読み込み、パーツ化
    with open(INPUT_CSV, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            
            words = []
            seen = set()
            
            def add_words(new_words, is_kimari=False):
                for w in new_words:
                    if w in seen: continue
                    if len(w) >= (1 if is_kimari else 2):
                        words.append(w)
                        seen.add(w)
                        
            add_words([kimari], is_kimari=True)
            add_words([gendai.replace(" ", "")])
            
            k_blocks = kanji.split()
            h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): add_words(extract_kanji_parts(k_blocks[i]))
                if i < len(h_blocks): add_words(extract_kana_parts(h_blocks[i], i))
                
            raw_cards.append({"num": num, "words": words, "kanji": kanji, "kana": kana})

    # 2. 全体で単語が何回（いくつの札に）登場するかカウント (逆引きマップ)
    tag_map = {}
    for c in raw_cards:
        for w in c["words"]:
            tag_map.setdefault(w, []).append(c["num"])

    # 3. 札ごとに重複ワードをカウントして保存用データを作成
    card_results = []
    for c in raw_cards:
        num = c["num"]
        total_words_count = len(c["words"])
        
        # 他の札と被っている単語だけを抽出
        overlap_words = [w for w in c["words"] if len(tag_map[w]) > 1]
        overlap_count = len(overlap_words)
        
        # 被っている割合（%）
        overlap_ratio = (overlap_count / total_words_count * 100) if total_words_count > 0 else 0.0
        
        card_results.append({
            "num": num,
            "kanji": c["kanji"],
            "total_count": total_words_count,
            "overlap_count": overlap_count,
            "overlap_ratio": overlap_ratio,
            "overlap_words_list": " ".join(overlap_words)
        })

    # 4. 重複ワード数が「多い順」にソート
    card_results.sort(key=lambda x: (x["overlap_count"], x["overlap_ratio"], x["num"]), reverse=True)

    # 5. CSVファイルに出力
    os.makedirs(os.path.dirname(OUTPUT_CARD_SUMMARY_CSV), exist_ok=True)
    
    with open(OUTPUT_CARD_SUMMARY_CSV, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        # ヘッダー
        writer.writerow([
            "札番号", 
            "漢字（冒頭）", 
            "総キーワード数", 
            "重複キーワード数", 
            "重複割合(%)", 
            "重複しているキーワード一覧"
        ])
        
        # ★ for文をしっかり with のインデントの中に配置しました
        for c in card_results:
            writer.writerow([
                c["num"],
                c["kanji"],
                c["total_count"],
                c["overlap_count"],
                f"{c['overlap_ratio']:.1f}%",
                c["overlap_words_list"]
            ])

    print(f"成功: 札ごとの重複集計を {OUTPUT_CARD_SUMMARY_CSV} に出力しました。")
    print(f"対象札総数: {len(card_results)} 枚")

if __name__ == "__main__":
    export_card_overlap_summary()

成功: 札ごとの重複集計を results/karuta_card_overlap_summary.csv に出力しました。
対象札総数: 101 枚


辞書作成方法改善。後半の区切り方改善。3句目の区切り少なく

In [9]:
import csv
import os

# 読み込み元ファイル（お使いの環境に合わせてパスを調整してください）
CSV_FILE = "data/karuta_data.csv"
# 辞書の中身を出力する確認用CSV
OUTPUT_DICT_CSV = "karuta_dictionary_check.csv"

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    
    if pos_index == 0:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    elif pos_index >= 2:
        p.append(text[0:2])
        if L >= 3:
            p.append(text[-3:])
            
    return p

def extract_kanji_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    
    if pos_index < 2:
        if L >= 2: p.append(text[:2])
        if L >= 3: p.append(text[:3])
        if L >= 2: p.append(text[-2:])
        if L >= 3: p.append(text[-3:])
    else:
        p.append(text[0:2])
        if L >= 3:
            p.append(text[-3:])
            
    return p

def export_dictionary_to_csv():
    if not os.path.exists(CSV_FILE):
        print(f"Error: {CSV_FILE} が見つかりません。")
        return

    rows_to_write = []

    with open(CSV_FILE, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            
            words = []
            seen = set()
            
            # 重複なく単語を追加していくための内部関数
            def add_words(new_words, origin_type, phrase_index=None):
                for w in new_words:
                    if w in seen: continue
                    # 決まり字なら1文字以上、それ以外は2文字以上
                    if len(w) >= (1 if origin_type == "決まり字" else 2):
                        seen.add(w)
                        # CSV出力用に情報をセット
                        context = origin_type
                        if phrase_index is not None:
                            context += f" ({phrase_index + 1}句目)"
                        rows_to_write.append([num, w, len(w), context])

            # 1. 決まり字の登録
            add_words([kimari], "決まり字")
            
            # 2. 現代語発音（全体）の登録
            add_words([gendai.replace(" ", "")], "現代語発音（全体）")
            
            # 3. 各句の漢字・ひらがなからの切り出し
            k_blocks = kanji.split()
            h_blocks = kana.split()
            
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): 
                    kanji_parts = extract_kanji_parts(k_blocks[i], i)
                    add_words(kanji_parts, "漢字パーツ", phrase_index=i)
                if i < len(h_blocks): 
                    kana_parts = extract_kana_parts(h_blocks[i], i)
                    add_words(kana_parts, "ひらがなパーツ", phrase_index=i)

    # 確認用CSVへの書き出し
    with open(OUTPUT_DICT_CSV, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        # ヘッダー行
        writer.writerow(["札番号", "登録キーワード", "文字数", "生成元の情報"])
        writer.writerows(rows_to_write)

    print(f"成功: 登録辞書の中身を 『{OUTPUT_DICT_CSV}』 に出力しました！")

if __name__ == "__main__":
    export_dictionary_to_csv()

成功: 登録辞書の中身を 『karuta_dictionary_check.csv』 に出力しました！


In [10]:
import csv
import os

# 読み込み元ファイル
CSV_FILE = "data/karuta_data.csv"
# 新しい確認用CSV（横並び・統計付き）
OUTPUT_SUMMARY_CSV = "karuta_card_summary.csv"

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    
    if pos_index == 0:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    elif pos_index >= 2:
        p.append(text[0:2])
        if L >= 3:
            p.append(text[-3:])
            
    return p

def extract_kanji_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    
    if pos_index < 2:
        if L >= 2: p.append(text[:2])
        if L >= 3: p.append(text[:3])
        if L >= 2: p.append(text[-2:])
        if L >= 3: p.append(text[-3:])
    else:
        p.append(text[0:2])
        if L >= 3:
            p.append(text[-3:])
            
    return p

def export_card_summary_to_csv():
    if not os.path.exists(CSV_FILE):
        print(f"Error: {CSV_FILE} が見つかりません。")
        return

    summary_rows = []

    with open(CSV_FILE, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            
            words = []
            seen = set()
            
            # メイン処理と同様の重複排除ロジック
            def add_words(new_words, is_kimari=False):
                for w in new_words:
                    if w in seen: continue
                    if len(w) >= (1 if is_kimari else 2):
                        words.append(w)
                        seen.add(w)

            # メイン処理と同じ順序・ルールで単語を蓄積
            add_words([kimari], is_kimari=True)
            add_words([gendai.replace(" ", "")])
            
            k_blocks = kanji.split()
            h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): add_words(extract_kanji_parts(k_blocks[i], i))
                if i < len(h_blocks): add_words(extract_kana_parts(h_blocks[i], i))
            
            # 各札の統計を計算
            word_count = len(words)               # 単語数
            total_char_count = sum(len(w) for w in words)  # 総文字数（満点スコア）
            words_space_separated = " ".join(words)  # 全単語を横一列に結合
            
            # CSVの1行分のデータを作成
            summary_rows.append([
                num, 
                kanji, 
                kana, 
                word_count, 
                total_char_count, 
                words_space_separated
            ])

    # 集計用CSVへ書き出し
    with open(OUTPUT_SUMMARY_CSV, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        # ヘッダー行
        writer.writerow([
            "札番号", 
            "元の漢字", 
            "元のひらがな", 
            "単語数", 
            "総文字数（満点）", 
            "登録キーワード一覧"
        ])
        writer.writerows(summary_rows)

    print(f"成功: 札ごとの統計辞書を 『{OUTPUT_SUMMARY_CSV}』 に出力しました！")

if __name__ == "__main__":
    export_card_summary_to_csv()

成功: 札ごとの統計辞書を 『karuta_card_summary.csv』 に出力しました！


In [15]:
import csv
import os

# 読み込み元ファイル
CSV_FILE = "data/karuta_data.csv"
# 新しい確認用CSV（句ごとに仕分け・トリミング・統計付き）
OUTPUT_SUMMARY_CSV = "karuta_card_summary_separated.csv"

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    
    # 1句目
    if pos_index == 0:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    # 2句目
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    # 3句目以降（3句目はアタマ2・末尾3のみに制限）
    elif pos_index >= 2:
        p.append(text[0:2])
        if L >= 3:
            p.append(text[-3:])
            
    return p

def extract_kanji_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    
    # 1句目・2句目
    if pos_index < 2:
        if L >= 2: p.append(text[:2])
        if L >= 3: p.append(text[:3])
        if L >= 2: p.append(text[-2:])
        if L >= 3: p.append(text[-3:])
    # 3句目以降（3句目はアタマ2・末尾3のみに制限）
    else:
        p.append(text[0:2])
        if L >= 3:
            p.append(text[-3:])
            
    return p

def export_separated_summary_to_csv():
    if not os.path.exists(CSV_FILE):
        print(f"Error: {CSV_FILE} が見つかりません。")
        return

    raw_cards = []

    # 1. まず全札の単語を生成元ごとに「構造化」して収集する
    with open(CSV_FILE, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            
            # 内部の重複チェック用
            seen = set()
            
            # 各パーツを格納する辞書（最終的にここから順番に取り出す）
            card_data = {
                "num": num,
                "kanji_orig": kanji,
                "kana_orig": kana,
                "決まり字": [],
                "現代語発音": [],
                "漢字1句目": [], "漢字2句目": [], "漢字3句目": [], "漢字4句目": [], "漢字5句目": [],
                "ひらがな1句目": [], "ひらがな2句目": [], "ひらがな3句目": [], "ひらがな4句目": [], "ひらがな5句目": [],
                "ordered_words": []  # トリミング判定用に、登録された順番の単語リスト
            }
            
            def add_word_to_zone(word, zone_key, is_kimari=False):
                if word in seen: return
                if len(word) >= (1 if is_kimari else 2):
                    seen.add(word)
                    card_data[zone_key].append(word)
                    card_data["ordered_words"].append(word)

            # 決まり字 & 現代語発音
            add_word_to_zone(kimari, "決まり字", is_kimari=True)
            add_word_to_zone(gendai.replace(" ", ""), "現代語発音")
            
            # 句ごとの処理 (通常5句まで)
            k_blocks = kanji.split()
            h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks):
                    k_parts = extract_kanji_parts(k_blocks[i], i)
                    for kp in k_parts:
                        add_word_to_zone(kp, f"漢字{i+1}句目")
                if i < len(h_blocks):
                    h_parts = extract_kana_parts(h_blocks[i], i)
                    for hp in h_parts:
                        add_word_to_zone(hp, f"ひらがな{i+1}句目")
                        
            raw_cards.append(card_data)

    # 2. トリミング用の最小単語数を特定
    min_word_count = min(len(c["ordered_words"]) for c in raw_cards)
    print(f"【判定結果】システム全体の最小単語数は 『{min_word_count}個』 です。")
    print(f"この数を超えた後ろのパーツは、各カテゴリから自動で除外されます。\n")

    # 3. 各札ごとに有効な単語（トリミング内）だけを仕分けて行を生成
    final_rows = []
    for c in raw_cards:
        # この札で「トリミング枠内」に残った単語のセットを作る
        allowed_words = set(c["ordered_words"][:min_word_count])
        
        # 枠内に残った単語だけで統計を再計算
        final_words = [w for w in c["ordered_words"] if w in allowed_words]
        word_count = len(final_words)
        total_char_count = sum(len(w) for w in final_words)
        
        # 各カテゴリのリストから、allowed_wordsに含まれるものだけをスペース区切りで抽出
        def get_valid_joined_words(zone_key):
            valid_list = [w for w in c[zone_key] if w in allowed_words]
            return " ".join(valid_list)

        # 横一列のデータ行を組み立て
        row_data = [
            c["num"],
            c["kanji_orig"],
            c["kana_orig"],
            word_count,
            total_char_count,
            get_valid_joined_words("決まり字"),
            get_valid_joined_words("現代語発音"),
            get_valid_joined_words("漢字1句目"),
            get_valid_joined_words("漢字2句目"),
            get_valid_joined_words("漢字3句目"),
            get_valid_joined_words("漢字4句目"),
            get_valid_joined_words("漢字5句目"),
            get_valid_joined_words("ひらがな1句目"),
            get_valid_joined_words("ひらがな2句目"),
            get_valid_joined_words("ひらがな3句目"),
            get_valid_joined_words("ひらがな4句目"),
            get_valid_joined_words("ひらがな5句目"),
        ]
        final_rows.append(row_data)

    # 4. 仕分け済みCSVへの書き出し
    headers = [
        "札番号", "元の漢字", "元のひらがな", "単語数", "総文字数（満点）",
        "決まり字", "現代語発音（全体）",
        "漢字パーツ(1句目)", "漢字パーツ(2句目)", "漢字パーツ(3句目)", "漢字パーツ(4句目)", "漢字パーツ(5句目)",
        "ひらがなパーツ(1句目)", "ひらがなパーツ(2句目)", "ひらがなパーツ(3句目)", "ひらがなパーツ(4句目)", "ひらがなパーツ(5句目)"
    ]

    with open(OUTPUT_SUMMARY_CSV, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(final_rows)

    print(f"成功: カテゴリ別に仕分けた統計辞書を 『{OUTPUT_SUMMARY_CSV}』 に出力しました！")

if __name__ == "__main__":
    export_separated_summary_to_csv()

【判定結果】システム全体の最小単語数は 『16個』 です。
この数を超えた後ろのパーツは、各カテゴリから自動で除外されます。

成功: カテゴリ別に仕分けた統計辞書を 『karuta_card_summary_separated.csv』 に出力しました！


In [16]:
import csv
import os

CSV_FILE = "data/karuta_data.csv"
OUTPUT_SUMMARY_CSV = "karuta_card_summary_trimmed.csv"

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    if pos_index == 0 or pos_index >= 2:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    return p

def extract_kanji_parts(text):
    text = text.replace(" ", "")
    p = []
    if len(text) >= 2: p.append(text[:2])
    if len(text) >= 3: p.append(text[:3])
    if len(text) >= 2: p.append(text[-2:])
    if len(text) >= 3: p.append(text[-3:])
    return p

def export_trimmed_summary_to_csv():
    if not os.path.exists(CSV_FILE):
        print(f"Error: {CSV_FILE} が見つかりません。")
        return

    raw_cards = []

    # 1. まず全札の単語リストを一旦生成する
    with open(CSV_FILE, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            
            words = []
            seen = set()
            
            def add_words(new_words, is_kimari=False):
                for w in new_words:
                    if w in seen: continue
                    if len(w) >= (1 if is_kimari else 2):
                        words.append(w)
                        seen.add(w)

            add_words([kimari], is_kimari=True)
            add_words([gendai.replace(" ", "")])
            
            k_blocks = kanji.split()
            h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): add_words(extract_kanji_parts(k_blocks[i]))
                if i < len(h_blocks): add_words(extract_kana_parts(h_blocks[i], i))
                
            raw_cards.append({
                "num": num, "kanji": kanji, "kana": kana, "words": words
            })

    # 2. 全札の中で「最も少ない単語数」をみつける（最小値の特定）
    min_word_count = min(len(c["words"]) for c in raw_cards)
    print(f"【判定結果】全100札の中での最小単語数は 『{min_word_count}個』 です。")
    print(f"これに合わせて、すべての札の登録単語を最大{min_word_count}個に制限（末尾カット）します。\n")

    # 3. 最小値に合わせて末尾をカットし、CSV用の行を作る
    summary_rows = []
    for c in raw_cards:
        # ★ここが肝：最小単語数に合わせて、リストの末尾からスライスして消す
        trimmed_words = c["words"][:min_word_count]
        
        word_count = len(trimmed_words)               # 当然すべて同じ（min_word_count）になる
        total_char_count = sum(len(w) for w in trimmed_words)  # カットされた後の総文字数
        words_space_separated = " ".join(trimmed_words)
        
        summary_rows.append([
            c["num"], 
            c["kanji"], 
            c["kana"], 
            word_count, 
            total_char_count, 
            words_space_separated
        ])

    # 集計用CSVへ書き出し
    with open(OUTPUT_SUMMARY_CSV, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "札番号", "元の漢字", "元のひらがな", "調整後単語数", "調整後総文字数（満点）", "登録キーワード一覧（トリミング後）"
        ])
        writer.writerows(summary_rows)

    print(f"成功: トリミング後の統計辞書を 『{OUTPUT_SUMMARY_CSV}』 に出力しました！")

if __name__ == "__main__":
    export_trimmed_summary_to_csv()

【判定結果】全100札の中での最小単語数は 『19個』 です。
これに合わせて、すべての札の登録単語を最大19個に制限（末尾カット）します。

成功: トリミング後の統計辞書を 『karuta_card_summary_trimmed.csv』 に出力しました！


In [19]:
import csv
import os

CSV_FILE = "data/karuta_data.csv"
OUTPUT_SUMMARY_CSV = "karuta_card_summary_final.csv"

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    
    if pos_index == 0:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    elif pos_index >= 2:
        p.append(text[0:2])
        if L >= 3: p.append(text[-3:])
            
    return p

def extract_kanji_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    
    if pos_index < 2:
        if L >= 2: p.append(text[:2])
        if L >= 3: p.append(text[:3])
        if L >= 2: p.append(text[-2:])
        if L >= 3: p.append(text[-3:])
    else:
        p.append(text[0:2])
        if L >= 3: p.append(text[-3:])
            
    return p

def export_final_summary_to_csv():
    if not os.path.exists(CSV_FILE):
        print(f"Error: {CSV_FILE} が見つかりません。")
        return

    raw_cards = []

    # 1. まず全札の単語を生成元グループごとに分けて収集（生成順を最優先）
    with open(CSV_FILE, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            
            seen = set()
            card_data = {
                "num": num, "kanji_orig": kanji, "kana_orig": kana,
                "決まり字": [], "現代語発音": [],
                "漢字1句目": [], "漢字2句目": [], "漢字3句目": [], "漢字4句目": [], "漢字5句目": [],
                "ひらがな1句目": [], "ひらがな2句目": [], "ひらがな3句目": [], "ひらがな4句目": [], "ひらがな5句目": [],
                "ordered_words": []  # トリミング順序を管理する全体プール
            }
            
            def add_word_to_zone(word, zone_key, is_kimari=False):
                if word in seen: return
                if len(word) >= (1 if is_kimari else 2):
                    seen.add(word)
                    card_data[zone_key].append(word)
                    card_data["ordered_words"].append(word)

            # 登録順にプールへ格納
            add_word_to_zone(kimari, "決まり字", is_kimari=True)
            add_word_to_zone(gendai.replace(" ", ""), "現代語発音")
            
            k_blocks = kanji.split()
            h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks):
                    for kp in extract_kanji_parts(k_blocks[i], i):
                        add_word_to_zone(kp, f"漢字{i+1}句目")
                if i < len(h_blocks):
                    for hp in extract_kana_parts(h_blocks[i], i):
                        add_word_to_zone(hp, f"ひらがな{i+1}句目")
                        
            raw_cards.append(card_data)

    # 2. 通常の札（1〜100番）のプールから最小単語数を割り出す
    normal_counts = [len(c["ordered_words"]) for c in raw_cards if c["num"] not in ["0", "00"]]
    min_word_count = min(normal_counts)
    
    print(f"【判定】1〜100番の最小単語数: {min_word_count} 個 に統一します。")
    print(f"【判定】0番（序歌）の許容単語数: {min_word_count * 2} 個（2倍）に設定します。\n")

    # 3. 割り出した統一単語数で一律末尾カットし、各句の列に仕分け
    final_rows = []
    for c in raw_cards:
        # 0番かそれ以外かでカットラインを決定
        limit = min_word_count * 2 if c["num"] in ["0", "00"] else min_word_count
        
        # 全体プールから一律で末尾カット（単語数を完全に統一）
        allowed_words = c["ordered_words"][:limit]
        allowed_set = set(allowed_words)
        
        # 統計（単語数は1〜100番なら全て完全に同じ数値になります）
        actual_count = len(allowed_words)
        total_chars = sum(len(w) for w in allowed_words)
        
        def get_joined_valid_words(zone_key):
            # カットされずに残った単語だけを、該当する句のセルに配置
            return " ".join([w for w in c[zone_key] if w in allowed_set])

        final_rows.append([
            c["num"], c["kanji_orig"], c["kana_orig"],
            actual_count, total_chars,
            get_joined_valid_words("決まり字"),
            get_joined_valid_words("現代語発音"),
            get_joined_valid_words("漢字1句目"), get_joined_valid_words("漢字2句目"), get_joined_valid_words("漢字3句目"), get_joined_valid_words("漢字4句目"), get_joined_valid_words("漢字5句目"),
            get_joined_valid_words("ひらがな1句目"), get_joined_valid_words("ひらがな2句目"), get_joined_valid_words("ひらがな3句目"), get_joined_valid_words("ひらがな4句目"), get_joined_valid_words("ひらがな5句目")
        ])

    # 4. CSV出力
    headers = [
        "札番号", "元の漢字", "元のひらがな", "調整後単語数", "調整後総文字数（満点）",
        "決まり字", "現代語発音（全体）",
        "漢字パーツ(1句目)", "漢字パーツ(2句目)", "漢字パーツ(3句目)", "漢字パーツ(4句目)", "漢字パーツ(5句目)",
        "ひらがなパーツ(1句目)", "ひらがなパーツ(2句目)", "ひらがなパーツ(3句目)", "ひらがなパーツ(4句目)", "ひらがなパーツ(5句目)"
    ]

    with open(OUTPUT_SUMMARY_CSV, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(final_rows)

    print(f"成功: 単語数を完全に統一した仕分けCSVを 『{OUTPUT_SUMMARY_CSV}』 に出力しました。")

if __name__ == "__main__":
    export_final_summary_to_csv()

【判定】1〜100番の最小単語数: 16 個 に統一します。
【判定】0番（序歌）の許容単語数: 32 個（2倍）に設定します。

成功: 単語数を完全に統一した仕分けCSVを 『karuta_card_summary_final.csv』 に出力しました。


1句目、3句目の区切り方同じ＆単語統一無し版

In [20]:
import csv
import os

# 元のシステムと同じ設定
CSV_FILE = "data/karuta_data.csv"
OUTPUT_SUMMARY_CSV = "karuta_card_summary_original_sync.csv"

# --- 元のシステムから完全移植（1文字目の特例やインデックス処理もそのまま） ---
def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    if pos_index == 0 or pos_index >= 2:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    return p

def extract_kanji_parts(text):
    text = text.replace(" ", "")
    p = []
    if len(text) >= 2: p.append(text[:2])
    if len(text) >= 3: p.append(text[:3])
    if len(text) >= 2: p.append(text[-2:])
    if len(text) >= 3: p.append(text[-3:])
    return p

def export_original_sync_csv():
    if not os.path.exists(CSV_FILE):
        print(f"Error: {CSV_FILE} が見つかりません。")
        return

    final_rows = []

    with open(CSV_FILE, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            
            # 各句・パーツ個別の格納用（元の登録順序を保つためリストで管理）
            card_zones = {
                "決まり字": [],
                "現代語発音": [],
                "漢字1句目": [], "漢字2句目": [], "漢字3句目": [], "漢字4句目": [], "漢字5句目": [],
                "ひらがな1句目": [], "ひらがな2句目": [], "ひらがな3句目": [], "ひらがな4句目": [], "ひらがな5句目": [],
            }
            
            seen = set()
            words_pool = []  # 札ごとの全有効単語（重複なし）
            
            # 元の add_words の重複排除ルールと文字数制限を完全に再現
            def add_word_to_zone(word, zone_key, is_kimari=False):
                if word in seen: return
                if len(word) >= (1 if is_kimari else 2):
                    seen.add(word)
                    card_zones[zone_key].append(word)
                    words_pool.append(word)

            # 1. 決まり字 と 現代語発音 の登録
            add_word_to_zone(kimari, "決まり字", is_kimari=True)
            add_word_to_zone(gendai.replace(" ", ""), "現代語発音")
            
            # 2. 漢字・ひらがなブロックの分解と登録（元のループ構造と完全一致）
            k_blocks = kanji.split()
            h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): 
                    # 元のシステムはextract_kanji_partsにi（インデックス）を渡さない
                    for kp in extract_kanji_parts(k_blocks[i]):
                        add_word_to_zone(kp, f"漢字{i+1}句目")
                if i < len(h_blocks): 
                    for hp in extract_kana_parts(h_blocks[i], i):
                        add_word_to_zone(hp, f"ひらがな{i+1}句目")
            
            # 3. 統計の計算（元のシステムの満点スコア計算と完全に一致）
            word_count = len(words_pool)
            total_char_count = sum(len(w) for w in words_pool)
            
            # 4. CSV用の行データ組み立て
            row_data = [
                num,
                kanji,
                kana,
                word_count,        # 元のシステムにおける実際の単語数
                total_char_count,  # 元のシステムにおける実際の満点スコア（card_max_scores[num]）
                " ".join(card_zones["決まり字"]),
                " ".join(card_zones["現代語発音"]),
                " ".join(card_zones["漢字1句目"]),
                " ".join(card_zones["漢字2句目"]),
                " ".join(card_zones["漢字3句目"]),
                " ".join(card_zones["漢字4句目"]),
                " ".join(card_zones["漢字5句目"]),
                " ".join(card_zones["ひらがな1句目"]),
                " ".join(card_zones["ひらがな2句目"]),
                " ".join(card_zones["ひらがな3句目"]),
                " ".join(card_zones["ひらがな4句目"]),
                " ".join(card_zones["ひらがな5句目"]),
            ]
            final_rows.append(row_data)

    # 5. CSVファイルへの書き出し
    headers = [
        "札番号", "元の漢字", "元のひらがな", "単語数(元システム)", "満点スコア(元システム)",
        "決まり字", "現代語発音（全体）",
        "漢字パーツ(1句目)", "漢字パーツ(2句目)", "漢字パーツ(3句目)", "漢字パーツ(4句目)", "漢字パーツ(5句目)",
        "ひらがなパーツ(1句目)", "ひらがなパーツ(2句目)", "ひらがなパーツ(3句目)", "ひらがなパーツ(4句目)", "ひらがなパーツ(5句目)"
    ]

    with open(OUTPUT_SUMMARY_CSV, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(final_rows)

    print(f"成功: 元のリアルタイムコードと完全に同期した仕分けCSVを 『{OUTPUT_SUMMARY_CSV}』 に出力しました！")

if __name__ == "__main__":
    export_original_sync_csv()

成功: 元のリアルタイムコードと完全に同期した仕分けCSVを 『karuta_card_summary_original_sync.csv』 に出力しました！


1句目、3句目同じ区切り方＆19単語統一　元の奴

In [21]:
import csv
import os

CSV_FILE = "data/karuta_data.csv"
OUTPUT_SUMMARY_CSV = "karuta_card_summary_orig_trimmed.csv"

# --- 元のシステムの区切り方を完全再現 ---
def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    # 元のルール：1句目と3句目以降が同じ処理
    if pos_index == 0 or pos_index >= 2:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    return p

def extract_kanji_parts(text):
    text = text.replace(" ", "")
    p = []
    if len(text) >= 2: p.append(text[:2])
    if len(text) >= 3: p.append(text[:3])
    if len(text) >= 2: p.append(text[-2:])
    if len(text) >= 3: p.append(text[-3:])
    return p

def export_orig_trimmed_csv():
    if not os.path.exists(CSV_FILE):
        print(f"Error: {CSV_FILE} が見つかりません。")
        return

    raw_cards = []

    # 1. 元の区切り方で、全札の単語を生成順にプール
    with open(CSV_FILE, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            
            seen = set()
            card_data = {
                "num": num, "kanji_orig": kanji, "kana_orig": kana,
                "決まり字": [], "現代語発音": [],
                "漢字1句目": [], "漢字2句目": [], "漢字3句目": [], "漢字4句目": [], "漢字5句目": [],
                "ひらがな1句目": [], "ひらがな2句目": [], "ひらがな3句目": [], "ひらがな4句目": [], "ひらがな5句目": [],
                "ordered_words": []  # トリミング順序管理用
            }
            
            def add_word_to_zone(word, zone_key, is_kimari=False):
                if word in seen: return
                if len(word) >= (1 if is_kimari else 2):
                    seen.add(word)
                    card_data[zone_key].append(word)
                    card_data["ordered_words"].append(word)

            add_word_to_zone(kimari, "決まり字", is_kimari=True)
            add_word_to_zone(gendai.replace(" ", ""), "現代語発音")
            
            k_blocks = kanji.split()
            h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks):
                    for kp in extract_kanji_parts(k_blocks[i]):
                        add_word_to_zone(kp, f"漢字{i+1}句目")
                if i < len(h_blocks):
                    for hp in extract_kana_parts(h_blocks[i], i):
                        add_word_to_zone(hp, f"ひらがな{i+1}句目")
                        
            raw_cards.append(card_data)

    # 2. 元の区切り方をした状態での「最小単語数」を割り出す
    normal_counts = [len(c["ordered_words"]) for c in raw_cards if c["num"] not in ["0", "00"]]
    min_word_count = min(normal_counts)
    
    print(f"【元の区切り方での判定】")
    print(f" ➔ 1〜100番の最小単語数: {min_word_count} 個 に統一します。")
    print(f" ➔ 0番（序歌）の許容単語数: {min_word_count * 2} 個（2倍）に設定します。\n")

    # 3. 割り出した最小単語数で末尾トリミングし、仕分け
    final_rows = []
    for c in raw_cards:
        limit = min_word_count * 2 if c["num"] in ["0", "00"] else min_word_count
        
        # 全体プールから一律で末尾カット
        allowed_words = c["ordered_words"][:limit]
        allowed_set = set(allowed_words)
        
        actual_count = len(allowed_words)
        total_chars = sum(len(w) for w in allowed_words)
        
        def get_joined_valid_words(zone_key):
            return " ".join([w for w in c[zone_key] if w in allowed_set])

        final_rows.append([
            c["num"], c["kanji_orig"], c["kana_orig"],
            actual_count, total_chars,
            get_joined_valid_words("決まり字"),
            get_joined_valid_words("現代語発音"),
            get_joined_valid_words("漢字1句目"), get_joined_valid_words("漢字2句目"), get_joined_valid_words("漢字3句目"), get_joined_valid_words("漢字4句目"), get_joined_valid_words("漢字5句目"),
            get_joined_valid_words("ひらがな1句目"), get_joined_valid_words("ひらがな2句目"), get_joined_valid_words("ひらがな3句目"), get_joined_valid_words("ひらがな4句目"), get_joined_valid_words("ひらがな5句目")
        ])

    # 4. CSV出力
    headers = [
        "札番号", "元の漢字", "元のひらがな", "統一単語数", "総文字数（満点）",
        "決まり字", "現代語発音（全体）",
        "漢字パーツ(1句目)", "漢字パーツ(2句目)", "漢字パーツ(3句目)", "漢字パーツ(4句目)", "漢字パーツ(5句目)",
        "ひらがなパーツ(1句目)", "ひらがなパーツ(2句目)", "ひらがなパーツ(3句目)", "ひらがなパーツ(4句目)", "ひらがなパーツ(5句目)"
    ]

    with open(OUTPUT_SUMMARY_CSV, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(final_rows)

    print(f"成功: 元の区切り方のまま最小単語数に統一したCSVを 『{OUTPUT_SUMMARY_CSV}』 に出力しました。")

if __name__ == "__main__":
    export_orig_trimmed_csv()

【元の区切り方での判定】
 ➔ 1〜100番の最小単語数: 19 個 に統一します。
 ➔ 0番（序歌）の許容単語数: 38 個（2倍）に設定します。

成功: 元の区切り方のまま最小単語数に統一したCSVを 『karuta_card_summary_orig_trimmed.csv』 に出力しました。


In [22]:
import csv
import os

CSV_FILE = "data/karuta_data.csv"
OUTPUT_SUMMARY_CSV = "karuta_card_summary_manual_fixed.csv"

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    
    # 新しい区切り方（3句目はアタマ2・末尾3）
    if pos_index == 0:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    elif pos_index >= 2:
        p.append(text[0:2])
        if L >= 3: p.append(text[-3:])
    return p

def extract_kanji_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    
    if pos_index < 2:
        if L >= 2: p.append(text[:2])
        if L >= 3: p.append(text[:3])
        if L >= 2: p.append(text[-2:])
        if L >= 3: p.append(text[-3:])
    else:
        p.append(text[0:2])
        if L >= 3: p.append(text[-3:])
    return p

def export_manual_fixed_csv():
    if not os.path.exists(CSV_FILE):
        print(f"Error: {CSV_FILE} が見つかりません。")
        return

    raw_cards = []

    with open(CSV_FILE, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num = row[0].strip()
            kanji = row[1].strip()
            kana = row[2].strip()
            kimari = row[3].strip()
            gendai = row[4].strip()
            
            # 6列目に手動追加単語があれば取得、なければ空
            manual_input = row[5].strip() if len(row) >= 6 else ""
            manual_words = manual_input.split() if manual_input else []
            
            seen = set()
            card_data = {
                "num": num, "kanji_orig": kanji, "kana_orig": kana,
                "決まり字": [], "手動追加": [], "現代語発音": [],
                "漢字1句目": [], "漢字2句目": [], "漢字3句目": [], "漢字4句目": [], "漢字5句目": [],
                "ひらがな1句目": [], "ひらがな2句目": [], "ひらがな3句目": [], "ひらがな4句目": [], "ひらがな5句目": [],
                "must_keep_words": [],  # 絶対に削らないプール（決まり字＋手動）
                "cuttable_words": []    # 最小単語数に合わせて削る対象プール（現代語＋自動パーツ）
            }
            
            # A. 絶対に削らないグループの登録
            # 決まり字
            if kimari and kimari not in seen:
                seen.add(kimari)
                card_data["決まり字"].append(kimari)
                card_data["must_keep_words"].append(kimari)
            # 手動追加単語
            for mw in manual_words:
                if mw and mw not in seen:
                    seen.add(mw)
                    card_data["手動追加"].append(mw)
                    card_data["must_keep_words"].append(mw)

            # B. 削るかもしれないグループの登録
            def add_to_cuttable(word, zone_key):
                if word in seen: return
                if len(word) >= 2:
                    seen.add(word)
                    card_data[zone_key].append(word)
                    card_data["cuttable_words"].append(word)

            # 現代語発音
            add_to_cuttable(gendai.replace(" ", ""), "現代語発音")
            
            # 各句の自動切り出し
            k_blocks = kanji.split()
            h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks):
                    for kp in extract_kanji_parts(k_blocks[i], i):
                        add_to_cuttable(kp, f"漢字{i+1}句目")
                if i < len(h_blocks):
                    for hp in extract_kana_parts(h_blocks[i], i):
                        add_to_cuttable(hp, f"ひらがな{i+1}句目")
                        
            raw_cards.append(card_data)

    # 2. 通常札（1〜100番）の「(強制生存) ＋ (削る候補)」の総数の最小値を割り出す
    normal_counts = [len(c["must_keep_words"]) + len(c["cuttable_words"]) for c in raw_cards if c["num"] not in ["0", "00"]]
    min_word_count = min(normal_counts)
    
    print(f"【手動単語追加ロジックでの判定】")
    print(f" ➔ 通常札の統一単語数: {min_word_count} 個")
    print(f" ➔ 0番札の許容単語数: {min_word_count * 2} 個\n")

    # 3. 各札をターゲット単語数に調整して仕分け
    final_rows = []
    for c in raw_cards:
        limit = min_word_count * 2 if c["num"] in ["0", "00"] else min_word_count
        
        # 強制生存分を引いた「残りの空き枠」を計算
        keep_len = len(c["must_keep_words"])
        allowed_cuttable_len = max(0, limit - keep_len)
        
        # 削る候補のリストから、空き枠の分だけ上から残す（あふれた末尾はカット）
        allowed_cuttable_words = c["cuttable_words"][:allowed_cuttable_len]
        
        # この札で最終的に生き残ったすべての単語の集合（セット）
        final_allowed_set = set(c["must_keep_words"] + allowed_cuttable_words)
        
        actual_count = len(final_allowed_set)
        total_chars = sum(len(w) for w in final_allowed_set)
        
        def get_joined_words(zone_key):
            return " ".join([w for w in c[zone_key] if w in final_allowed_set])

        final_rows.append([
            c["num"], c["kanji_orig"], c["kana_orig"],
            actual_count, total_chars,
            get_joined_words("決まり字"),
            get_joined_words("手動追加"),
            get_joined_words("現代語発音"),
            get_joined_words("漢字1句目"), get_joined_words("漢字2句目"), get_joined_words("漢字3句目"), get_joined_words("漢字4句目"), get_joined_words("漢字5句目"),
            get_joined_words("ひらがな1句目"), get_joined_words("ひらがな2句目"), get_joined_words("ひらがな3句目"), get_joined_words("ひらがな4句目"), get_joined_words("ひらがな5句目")
        ])

    # 4. CSV出力
    headers = [
        "札番号", "元の漢字", "元のひらがな", "調整後単語数", "調整後総文字数",
        "決まり字", "手動追加単語", "現代語発音（全体）",
        "漢字パーツ(1句目)", "漢字パーツ(2句目)", "漢字パーツ(3句目)", "漢字パーツ(4句目)", "漢字パーツ(5句目)",
        "ひらがなパーツ(1句目)", "ひらがなパーツ(2句目)", "ひらがなパーツ(3句目)", "ひらがなパーツ(4句目)", "ひらがなパーツ(5句目)"
    ]

    with open(OUTPUT_SUMMARY_CSV, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(final_rows)

    print(f"成功: 手動追加対応＆最小単語数統一のCSVを 『{OUTPUT_SUMMARY_CSV}』 に出力しました。")

if __name__ == "__main__":
    export_manual_fixed_csv()

【手動単語追加ロジックでの判定】
 ➔ 通常札の統一単語数: 17 個
 ➔ 0番札の許容単語数: 34 個

成功: 手動追加対応＆最小単語数統一のCSVを 『karuta_card_summary_manual_fixed.csv』 に出力しました。
